In [2]:
import os
import glob

import pandas as pd
import talib
from talib import MA_Type

In [3]:
def read_csv_files(nation, market, tickers=None):
    base_path = os.path.abspath(f'./stock_data/{nation}/{market}')
    all_files = glob.glob(os.path.join(base_path, "*.csv"))

    # tickers가 주어지면 해당 ticker만 필터링
    if tickers is not None:
        tickers = set(tickers)
        all_files = [f for f in all_files if os.path.splitext(os.path.basename(f))[0] in tickers]
        
    # 데이터 없는 파일 처리
    no_date_files = []
    for file_path in all_files:
        # ticker = os.path.splitext(os.path.basename(file_path))[0]
        try:
            # 헤더만 읽어서 컬럼명 확인
            df = pd.read_csv(file_path, nrows=1)
            if 'Date' not in df.columns:
                no_date_files.append(file_path)
        except Exception as e:
            print(f"{file_path} 읽기 오류: {e}")
        
    if no_date_files:
        print("아래 파일들은 데이터가 없습니다:")
        for file in no_date_files:
            print("-", os.path.basename(file))

    ticker_list = []
    df_list = []
    for file_path in all_files:
        if file_path in no_date_files:
            continue  # 데이터 없는 파일은 건너뜀
        ticker = os.path.splitext(os.path.basename(file_path))[0]
        ticker_list.append(ticker)
        df = pd.read_csv(file_path, parse_dates=['Date'])
        df_list.append(df)

    return ticker_list, df_list

In [3]:
# def mdd_from_price(adj_close, as_percent=False):
#     roll_max = adj_close.cummax()
#     dd = (adj_close / roll_max) - 1.0
#     mdd = dd.min()  # 음수 값(최대 낙폭)
#     return (-mdd * 100.0) if as_percent else (-mdd)

# def mdd_from_nav(nav: pd.Series, as_percent: bool = False):
#     # 전략의 누적자산곡선(NAV) 기준 MDD
#     roll_max = nav.cummax()
#     dd = (nav / roll_max) - 1.0
#     mdd = dd.min()
#     return (-mdd * 100.0) if as_percent else (-mdd)

In [4]:
def create_features(df):
    df["SMA_60"] = talib.SMA(df["Adj Close"], timeperiod=60)
    df["EMA_60"] = talib.EMA(df["Adj Close"], timeperiod=60)
    df["RSI_14"] = talib.RSI(df["Adj Close"], timeperiod=14)
    # macd_line, macd_signal, macd_hist = talib.MACD(df["Adj Close"], 12, 26, 9)
    # df["MACD"] = macd_line
    # df["MACD_signal"] = macd_signal
    # df["MACD_hist"] = macd_hist
    # u, m, l = talib.BBANDS(df["Adj Close"], timeperiod=20, nbdevup=2, nbdevdn=2, 
    #                        matype=MA_Type.SMA)
    # df["Bandwidth"] = (u - l) / m   # 정규화 밴드폭
    # df["BB_mid"], df["BB_upper"], df["BB_lower"] = m, u, l
    df["ATR_14"] = talib.ATR(df["High"], df["Low"], df["Close"], timeperiod=14)
    df["OBV"] = talib.OBV(df["Close"], df["Volume"])
    # df["MDD"] = mdd_from_price(df["Adj Close"])
    df = df.dropna()

    return df

In [4]:
def new_file_create(folder_name, ticker, df):
    df.to_csv(f'{folder_name}/{ticker}.csv', index=False)
    
    # return "생성이 완료되었습니다."

## 대한민국

In [7]:
kospi_ticker_list, kospi_df_list = read_csv_files('대한민국', '코스피 200 추종 ETF')

In [10]:
for i in range(len(kospi_df_list)):
    if len(kospi_ticker_list) != len(kospi_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    kospi_df = create_features(kospi_df_list[i])
    new_file_create("new_KOSPI 200", kospi_ticker_list[i], kospi_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### KOSPI

In [91]:
kospi_ticker_list, kospi_df_list = read_csv_files('대한민국', 'KOSPI')

In [92]:
for i in range(len(kospi_df_list)):
    if len(kospi_ticker_list) != len(kospi_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    kospi_df = create_features(kospi_df_list[i])
    new_file_create("new_KOSPI", kospi_ticker_list[i], kospi_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### KOSDAQ

In [93]:
kosdaq_ticker_list, kosdaq_df_list = read_csv_files('대한민국', 'KOSDAQ')

In [94]:
for i in range(len(kosdaq_df_list)):
    if len(kosdaq_ticker_list) != len(kosdaq_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    kosdaq_df = create_features(kosdaq_df_list[i])
    new_file_create("new_KOSDAQ", kosdaq_ticker_list[i], kosdaq_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


## 미국

In [11]:
nasdaq_ticker_list, nasdaq_df_list = read_csv_files('미국', 'NASDAQ 100 추종 ETF')

In [14]:
nasdaq_ticker_list, nasdaq_df_list = read_csv_files('미국', 'S&P 500 추종 ETF')

In [16]:
nasdaq_ticker_list, nasdaq_df_list = read_csv_files('미국', '다우존스지수 추종 ETF')

In [17]:
for i in range(len(nasdaq_df_list)):
    if len(nasdaq_ticker_list) != len(nasdaq_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    nasdaq_df = create_features(nasdaq_df_list[i])
    new_file_create("new_다우존스지수", nasdaq_ticker_list[i], nasdaq_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### NASDAQ

In [95]:
nasdaq_ticker_list, nasdaq_df_list = read_csv_files('미국', 'NASDAQ')

In [96]:
for i in range(len(nasdaq_df_list)):
    if len(nasdaq_ticker_list) != len(nasdaq_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    nasdaq_df = create_features(nasdaq_df_list[i])
    new_file_create("new_NASDAQ", nasdaq_ticker_list[i], nasdaq_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### NYSE

In [97]:
nyse_ticker_list, nyse_df_list = read_csv_files('미국', 'NYSE')

In [98]:
for i in range(len(nyse_df_list)):
    if len(nyse_ticker_list) != len(nyse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    nyse_df = create_features(nyse_df_list[i])
    new_file_create("new_NYSE", nyse_ticker_list[i], nyse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


## 일본

In [18]:
tse_ticker_list, tse_df_list = read_csv_files('일본', 'TOPIX 추종 ETF')

In [20]:
tse_ticker_list, tse_df_list = read_csv_files('일본', '니케이 225 추종 ETF')

In [21]:
for i in range(len(tse_df_list)):
    if len(tse_ticker_list) != len(tse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    tse_df = create_features(tse_df_list[i])
    new_file_create("new_NIKKEI 225", tse_ticker_list[i], tse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### TSE

In [99]:
tse_ticker_list, tse_df_list = read_csv_files('일본', 'TSE')

In [100]:
for i in range(len(tse_df_list)):
    if len(tse_ticker_list) != len(tse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    tse_df = create_features(tse_df_list[i])
    new_file_create("new_TSE", tse_ticker_list[i], tse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


## 중국

In [22]:
sse_ticker_list, sse_df_list = read_csv_files('중국', 'CSI 300 추종 ETF')

In [23]:
for i in range(len(sse_df_list)):
    if len(sse_ticker_list) != len(sse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    sse_df = create_features(sse_df_list[i])
    new_file_create("new_CSI 300", sse_ticker_list[i], sse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### SSE

In [101]:
sse_ticker_list, sse_df_list = read_csv_files('중국', 'SSE')

In [102]:
for i in range(len(sse_df_list)):
    if len(sse_ticker_list) != len(sse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    sse_df = create_features(sse_df_list[i])
    new_file_create("new_SSE", sse_ticker_list[i], sse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


## 유럽

In [24]:
euro_ticker_list, euro_df_list = read_csv_files('유럽', 'EURO STOXX 50 추종 ETF')

In [26]:
euro_ticker_list, euro_df_list = read_csv_files('유럽', 'MSCI Europe 추종 ETF')

In [28]:
euro_ticker_list, euro_df_list = read_csv_files('유럽', 'STOXX Europe 600 추종 ETF')

In [29]:
for i in range(len(euro_ticker_list)):
    if len(euro_ticker_list) != len(euro_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    euro_df = create_features(euro_df_list[i])
    new_file_create("new_STOXX Europe 600", euro_ticker_list[i], euro_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### LSE

In [103]:
lse_ticker_list, lse_df_list = read_csv_files('유럽', 'LSE')

In [104]:
for i in range(len(lse_df_list)):
    if len(lse_ticker_list) != len(lse_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    lse_df = create_features(lse_df_list[i])
    new_file_create("new_LSE", lse_ticker_list[i], lse_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### PAR

In [105]:
par_ticker_list, par_df_list = read_csv_files('유럽', 'PAR')

In [106]:
for i in range(len(par_df_list)):
    if len(par_ticker_list) != len(par_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    par_df = create_features(par_df_list[i])
    new_file_create("new_PAR", par_ticker_list[i], par_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.


### FRA

In [6]:
fra_ticker_list, fra_df_list = read_csv_files('유럽', 'FRA')

In [7]:
for i in range(len(fra_df_list)):
    if len(fra_ticker_list) != len(fra_df_list):
        print("티커 수와 데이터프레임 수가 일치하지 않습니다.")
        break
    fra_df = create_features(fra_df_list[i])
    new_file_create("new_FRA", fra_ticker_list[i], fra_df)
    
print("생성이 완료되었습니다.")

생성이 완료되었습니다.
